# Falarosa free lip-sync (LatentSync 1.5)

Puts a new voice (for example your Microsoft Ubax recording) onto a video of a person talking, and moves their lips to match.

**How to use, every time:**
1. **Runtime → Change runtime type → T4 GPU** (free), then **Save**.
2. **Runtime → Run all**. Allow Google Drive access when asked.
3. When the upload box appears, upload the **video** first, then the **audio** (MP3 or WAV).
4. The finished video downloads at the end, and a copy is saved in Google Drive under `falarosa-lipsync/outputs/`.

The first run takes about 15 minutes (it saves about 5 GB of model files to your Drive). Later runs take about 5 minutes to set up, then about 1–3 minutes per 20-second clip.

Everything is pinned to fixed versions so it keeps working when Colab updates.

In [ ]:
#@title 1. Check the GPU
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU. Go to Runtime > Change runtime type > T4 GPU, Save, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

In [ ]:
#@title 2. Connect Google Drive (model files are saved here once)
from google.colab import drive
drive.mount("/content/drive")
import os
CACHE = "/content/drive/MyDrive/falarosa-lipsync"
for d in ("checkpoints", "hf-cache", "outputs"):
    os.makedirs(f"{CACHE}/{d}", exist_ok=True)
print("Drive folder:", CACHE)

In [ ]:
#@title 3. Install a fixed Python 3.10 and the exact package versions (about 5 minutes)
import subprocess
script = r"""
set -e
MM=/content/mm
if [ ! -x $MM/envs/ls/bin/python ]; then
  mkdir -p $MM && cd $MM
  curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/1.5.10 | tar -xj bin/micromamba
  export MAMBA_ROOT_PREFIX=$MM
  ./bin/micromamba create -y -q -n ls -c conda-forge python=3.10.13 > /dev/null
fi
cd /content
if [ ! -d LatentSync ]; then
  git clone -q https://github.com/bytedance/LatentSync.git
fi
cd LatentSync && git checkout -q 7b380d6adde7aa23cba4a53bb15cd5cfb3c168e7
$MM/envs/ls/bin/python -m pip install -q --upgrade "pip==24.2" "setuptools==70.0.0" wheel cython
$MM/envs/ls/bin/python -m pip install -q -r requirements.txt
apt-get -qq install -y libgl1 ffmpeg > /dev/null
echo "Installed: $($MM/envs/ls/bin/python --version), LatentSync 1.5 (commit 7b380d6)"
"""
r = subprocess.run(["bash", "-c", script])
if r.returncode != 0:
    raise SystemExit("This step failed. Copy the error text above and send it to Claude.")


In [ ]:
#@title 4. Get the model files (downloads once, then reused from Drive)
import subprocess
script = r"""
set -e
CACHE=/content/drive/MyDrive/falarosa-lipsync
cd /content/LatentSync
rm -rf checkpoints && ln -s $CACHE/checkpoints checkpoints
export HF_HOME=$CACHE/hf-cache
HFCLI=/content/mm/envs/ls/bin/huggingface-cli
[ -f checkpoints/whisper/tiny.pt ] || $HFCLI download ByteDance/LatentSync-1.5 whisper/tiny.pt --local-dir checkpoints
[ -f checkpoints/latentsync_unet.pt ] || $HFCLI download ByteDance/LatentSync-1.5 latentsync_unet.pt --local-dir checkpoints
ls -la checkpoints checkpoints/whisper
"""
r = subprocess.run(["bash", "-c", script])
if r.returncode != 0:
    raise SystemExit("This step failed. Copy the error text above and send it to Claude.")


In [ ]:
#@title 5. Upload the video, then the audio
from google.colab import files
import shutil, os
os.makedirs("/content/in", exist_ok=True)
print("Upload the VIDEO (mp4):")
v = files.upload()
print("Upload the AUDIO (mp3 or wav):")
a = files.upload()
VIDEO = "/content/in/" + list(v)[0]; shutil.move(list(v)[0], VIDEO)
AUDIO = "/content/in/" + list(a)[0]; shutil.move(list(a)[0], AUDIO)
print("Video:", VIDEO); print("Audio:", AUDIO)

In [ ]:
#@title 6. Lip-sync (about 1–3 minutes for a 20-second clip)
import subprocess, os, time
os.chdir("/content/LatentSync")
env = dict(os.environ, HF_HOME="/content/drive/MyDrive/falarosa-lipsync/hf-cache")
# LatentSync wants 16 kHz mono WAV audio
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", AUDIO, "-ac", "1", "-ar", "16000", "/content/in/audio16k.wav"], check=True)
name = os.path.splitext(os.path.basename(VIDEO))[0] + "-lipsync-" + time.strftime("%Y%m%d-%H%M%S") + ".mp4"
OUT = "/content/" + name
cmd = ["/content/mm/envs/ls/bin/python", "-m", "scripts.inference",
       "--unet_config_path", "configs/unet/stage2.yaml",
       "--inference_ckpt_path", "checkpoints/latentsync_unet.pt",
       "--inference_steps", "20", "--guidance_scale", "2.0", "--seed", "1247",
       "--video_path", VIDEO, "--audio_path", "/content/in/audio16k.wav", "--video_out_path", OUT]
r = subprocess.run(cmd, env=env)
if r.returncode != 0 or not os.path.exists(OUT):
    raise SystemExit("Lip-sync failed. Copy the red error text above and send it to Claude.")
# put the original-quality audio back on the result
FINAL = "/content/drive/MyDrive/falarosa-lipsync/outputs/" + name
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", OUT, "-i", AUDIO, "-map", "0:v", "-map", "1:a", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest", FINAL], check=True)
print("Saved to Drive:", FINAL)

In [ ]:
#@title 7. Download the result
from google.colab import files
files.download(FINAL)